# Xuất tập test thành CSV đầu vào NIDR

Notebook chỉ đọc **output test đã chuẩn bị** của data run `20261021_052521_96f61f19`, tương ứng model **XGB_d4_n200** đã chốt. Không lấy train/validation, không chia lại tập, không cân bằng lớp, không huấn luyện hoặc fit tiền xử lý.

**Phân biệt hai bước:** test đã qua làm sạch, ánh xạ nhãn và kiểm soát split ở notebook processing. File `X_test_unprocessed.npy` vẫn giữ 46 đặc trưng **thang đo gốc**, float32, NaN được giữ lại. Đây là đầu vào đúng cho BE: bộ imputer/scaler đã đóng băng trong bundle sẽ tự `transform` khi dự đoán. Không xuất dữ liệu đã scale/impute để tránh tiền xử lý hai lần.

Mỗi CSV có **1–1.000 dòng dữ liệu**, không tính header, và **không quá 10 MiB (10.485.760 byte)** tính cả header UTF-8. Cột `label` tùy chọn chỉ để đối chiếu; không phải đầu vào model hay Gemini.

### Cách chạy trên Kaggle
1. Import notebook này. Chọn CPU; không cần GPU, Internet hoặc API key.
2. **Add Input → Notebook Output** của `ciciot2023-do-an-processing`, chọn phiên bản chứa `processed_data/20261021_052521_96f61f19/`.
3. Chỉnh cấu hình bên dưới nếu cần, rồi **Restart Session → Run All**.
4. Tải ZIP ở Output, giải nén và tải từng CSV trong `uploads/` vào NIDR. Không tải ZIP/manifest vào ô CSV.

Notebook xác minh checksum nguồn trước khi xuất và đọc lại từng CSV sau khi xuất. Nguồn sai thì dừng trước khi xuất. Nếu lỗi trong quá trình xuất, notebook ghi trạng thái `failed` và không phát hành ZIP thành công.

**Giới hạn đánh giá:** baseline validation/test đã từng được xem. Đây là dữ liệu test tái sử dụng để chạy/demo hệ thống, không phải một tập đánh giá độc lập mới. Nhãn trong CSV là nhãn 8 lớp từ dataset, không phải kết quả chuyên gia xác minh sự kiện thực tế.

In [ ]:
import csv
import hashlib
import io
import json
import platform
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

## 1. Cấu hình xuất

Mặc định xuất **toàn bộ test theo thứ tự đang lưu**, mỗi file tối đa 1.000 dòng. Để chạy thử nhanh, đặt `MAX_TOTAL_ROWS = 1000`; `START_ROW` là chỉ số từ 0 trong mảng test sau chuẩn bị. Để xuất không nhãn, đặt `INCLUDE_LABEL = False`.

`DATA_DIR = None` tự tìm đúng run trong `/kaggle/input`. Nếu có nhiều bản sao, điền đường dẫn thư mục run cụ thể. Không đổi run/checksum sang dữ liệu khác khi vẫn dùng model đã chốt.

In [ ]:
DATA_DIR = None
# Ví dụ đường dẫn (Kaggle có thể đặt tên mount khác):
# DATA_DIR = Path("/kaggle/input/notebooks/phungdinhdat/ciciot2023-do-an-processing/processed_data/20261021_052521_96f61f19")

INCLUDE_LABEL = True
ROWS_PER_FILE = 1000
MAX_FILE_BYTES = 10 * 1024 * 1024
START_ROW = 0
MAX_TOTAL_ROWS = None  # None: toàn bộ phần test còn lại; ví dụ 1000: chỉ lấy 1.000 dòng.
OUTPUT_ROOT = Path("/kaggle/working/nidr_test_uploads")
CREATE_ZIP = True

In [ ]:
# Hợp đồng lấy từ input_schema.json của integration_20261022_075217_0b67d48f.
FEATURE_NAMES = [
    "flow_duration",
    "Header_Length",
    "Protocol Type",
    "Duration",
    "Rate",
    "Srate",
    "Drate",
    "fin_flag_number",
    "syn_flag_number",
    "rst_flag_number",
    "psh_flag_number",
    "ack_flag_number",
    "ece_flag_number",
    "cwr_flag_number",
    "ack_count",
    "syn_count",
    "fin_count",
    "urg_count",
    "rst_count",
    "HTTP",
    "HTTPS",
    "DNS",
    "Telnet",
    "SMTP",
    "SSH",
    "IRC",
    "TCP",
    "UDP",
    "DHCP",
    "ARP",
    "ICMP",
    "IPv",
    "LLC",
    "Tot sum",
    "Min",
    "Max",
    "AVG",
    "Std",
    "Tot size",
    "IAT",
    "Number",
    "Magnitue",
    "Radius",
    "Covariance",
    "Variance",
    "Weight"
]
CLASS_NAMES = [
    "Normal",
    "BruteForce",
    "DDoS",
    "DoS",
    "Mirai",
    "Recon",
    "Spoofing",
    "Web-Based"
]
CLASS_TO_ID = {name: index for index, name in enumerate(CLASS_NAMES)}
PROTOCOL = "stacking_fixed_split_v1"
EXPECTED_DATA_RUN_ID = "20261021_052521_96f61f19"
# SHA256 được ghi trong source_artifacts.json của xgb_meta_20261021_092334_9024562d.
EXPECTED_MANIFEST_SHA256 = "f9c4cf30b1e9b62b9c2d95f804a24a94f4d5e77b85b5e2055ae1ceebdbff9553"
SYSTEM_MAX_ROWS = 1000
SYSTEM_MAX_BYTES = 10 * 1024 * 1024

## 2. Xác minh nguồn test

Chỉ đọc mảng `X_test_unprocessed.npy` và, khi cần nhãn, `y_test.npy`. Kiểm tra checksum đầy đủ có thể mất một lúc dù chỉ chọn xuất ít dòng. Không nạp model, pickle metadata hoặc các mảng train/validation.

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(path, data):
    Path(path).write_text(
        json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        while True:
            block = stream.read(4 * 1024 * 1024)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def find_data_dir(data_dir, input_root=Path("/kaggle/input")):
    if data_dir is not None:
        selected = Path(data_dir)
        if not selected.is_dir():
            raise FileNotFoundError(f"Không tìm thấy DATA_DIR: {selected}")
        return selected

    candidates = []
    for config_path in Path(input_root).rglob("preprocessing_config.json"):
        try:
            config = read_json(config_path)
        except (OSError, ValueError):
            continue
        if config.get("run_id") == EXPECTED_DATA_RUN_ID:
            candidates.append(config_path.parent)
    if len(candidates) != 1:
        raise ValueError(
            f"Cần đúng một output run {EXPECTED_DATA_RUN_ID}; tìm thấy: {candidates}. "
            "Gắn đúng output processing hoặc đặt DATA_DIR cụ thể."
        )
    return candidates[0]


def load_test_source(data_dir, include_label):
    data_dir = Path(data_dir)
    manifest_path = data_dir / "checksums.json"
    if sha256_file(manifest_path) != EXPECTED_MANIFEST_SHA256:
        raise ValueError("Checksum manifest khác nguồn đã chốt. Không xuất dữ liệu.")
    checksums = read_json(manifest_path)
    required = [
        "preprocessing_config.json", "feature_schema.json", "feature_names.txt",
        "X_test_unprocessed.npy",
    ]
    if include_label:
        required.append("y_test.npy")
    verified_checksums = {}
    for name in required:
        expected = checksums.get(name)
        actual = sha256_file(data_dir / name)
        if not isinstance(expected, str) or actual != expected:
            raise ValueError(f"Checksum không khớp: {name}")
        verified_checksums[name] = actual

    config = read_json(data_dir / "preprocessing_config.json")
    schema = read_json(data_dir / "feature_schema.json")
    status = read_json(data_dir / "run_status.json")
    for record in [config, status]:
        if record.get("run_id") != EXPECTED_DATA_RUN_ID or record.get("protocol") != PROTOCOL:
            raise ValueError("Run ID hoặc protocol nguồn không đúng.")
    if status.get("status") != "complete":
        raise ValueError("Processing nguồn chưa complete.")
    if schema.get("protocol") != PROTOCOL:
        raise ValueError("Protocol schema nguồn không đúng.")
    for record in [config, schema]:
        if record.get("preprocessing_fitted") is not False:
            raise ValueError("Không nhận test đã fit/transform tiền xử lý.")
        if record.get("class_names") != CLASS_NAMES or record.get("class_to_id") != CLASS_TO_ID:
            raise ValueError("Thứ tự hoặc ánh xạ 8 lớp không đúng.")
    feature_names = (data_dir / "feature_names.txt").read_text(encoding="utf-8").splitlines()
    if (feature_names != FEATURE_NAMES or config.get("features") != FEATURE_NAMES
            or schema.get("input_features") != FEATURE_NAMES
            or schema.get("output_features") != FEATURE_NAMES):
        raise ValueError("Tên/thứ tự 46 đặc trưng không đúng hợp đồng NIDR.")
    if schema.get("label_input_to_estimator") is not False:
        raise ValueError("Nhãn không được nằm trong đầu vào estimator.")
    test_record = config["artifacts"]["test"]
    if test_record.get("input_file") != "X_test_unprocessed.npy":
        raise ValueError("Chỉ cho phép X_test_unprocessed.npy.")
    if test_record.get("target_file") != "y_test.npy":
        raise ValueError("Nhãn phải lấy từ y_test.npy của cùng run.")

    values = np.load(data_dir / "X_test_unprocessed.npy", mmap_mode="r", allow_pickle=False)
    if (values.ndim != 2 or values.shape[1] != len(FEATURE_NAMES)
            or values.dtype != np.dtype("float32") or len(values) == 0
            or list(values.shape) != test_record.get("shape")):
        raise ValueError("Shape/dtype test không đúng manifest (cần float32, N × 46).")
    labels = None
    if include_label:
        labels = np.load(data_dir / "y_test.npy", mmap_mode="r", allow_pickle=False)
        if labels.shape != (len(values),) or not np.issubdtype(labels.dtype, np.integer):
            raise ValueError("y_test phải là vector số nguyên, cùng số dòng X_test.")
    source_info = {
        "run_id": EXPECTED_DATA_RUN_ID,
        "source_directory": str(data_dir),
        "split": "test",
        "input_file": "X_test_unprocessed.npy",
        "target_file": "y_test.npy" if include_label else None,
        "manifest_sha256": EXPECTED_MANIFEST_SHA256,
        "verified_checksums": verified_checksums,
        "shape": list(values.shape),
        "input_scale": "original_float32_nan_retained",
        "preprocessing_fitted_here": False,
        "previous_validation_test_viewed": True,
    }
    return values, labels, source_info

## 3. Chia CSV và kiểm tra đọc lại

Giới hạn dung lượng dùng **số byte UTF-8 thực tế**, gồm header và xuống dòng. Mỗi file đóng lại khi đạt giới hạn số dòng hoặc trước khi dòng tiếp theo làm vượt giới hạn byte. Phần dư cuối cùng vẫn được xuất.

NaN → ô trống; không điền median, không chuẩn hóa. Giá trị hữu hạn được ghi đủ chữ số để đọc lại float32 không đổi. Nếu còn Inf, notebook dừng vì dữ liệu này không phù hợp output processing đã chốt.

`manifest.csv` ghi đoạn chỉ số `[test_start_row, test_stop_row_exclusive)` của từng file để truy ngược về **mảng test sau chuẩn bị**, không phải số dòng CSV gốc trước làm sạch. NIDR đánh số dòng lại từ 0 trong mỗi CSV. Không thêm cột chỉ số vào CSV upload vì BE chỉ nhận 46 đặc trưng và `label` tùy chọn.

In [ ]:
def csv_line(fields):
    buffer = io.StringIO(newline="")
    csv.writer(buffer, lineterminator="\n").writerow(fields)
    return buffer.getvalue().encode("utf-8")


def validate_export_options(rows_per_file, max_file_bytes, start_row, max_total_rows, total_rows):
    if type(rows_per_file) is not int or not 1 <= rows_per_file <= SYSTEM_MAX_ROWS:
        raise ValueError("ROWS_PER_FILE phải là số nguyên từ 1 đến 1.000.")
    if type(max_file_bytes) is not int or not 1 <= max_file_bytes <= SYSTEM_MAX_BYTES:
        raise ValueError("MAX_FILE_BYTES phải là số nguyên không quá 10 MiB.")
    if type(start_row) is not int or not 0 <= start_row < total_rows:
        raise ValueError("START_ROW nằm ngoài tập test.")
    if max_total_rows is not None and (type(max_total_rows) is not int or max_total_rows < 1):
        raise ValueError("MAX_TOTAL_ROWS phải là None hoặc số nguyên dương.")
    stop_row = total_rows
    if max_total_rows is not None:
        stop_row = min(total_rows, start_row + max_total_rows)
    return stop_row


def validate_csv(path, expected_values, expected_labels, columns, max_file_bytes):
    if path.stat().st_size > max_file_bytes:
        raise ValueError(f"CSV vượt giới hạn byte: {path.name}")
    with path.open(newline="", encoding="utf-8") as stream:
        if next(csv.reader(stream)) != columns:
            raise ValueError(f"Header CSV không đúng: {path.name}")
    data = pd.read_csv(path)
    if list(data.columns) != columns or not 1 <= len(data) <= SYSTEM_MAX_ROWS:
        raise ValueError(f"Schema hoặc số dòng CSV không đúng: {path.name}")
    # Giống helper BE: pandas đọc float64 rồi chuyển 46 đặc trưng về float32.
    restored = data.loc[:, FEATURE_NAMES].astype(np.float64).to_numpy(dtype=np.float32)
    if not np.array_equal(restored, expected_values, equal_nan=True):
        raise ValueError(f"Giá trị/NaN/thứ tự dòng thay đổi khi đọc lại: {path.name}")
    if expected_labels is not None:
        names = [CLASS_NAMES[int(class_id)] for class_id in expected_labels]
        if data["label"].tolist() != names:
            raise ValueError(f"Nhãn không khớp y_test: {path.name}")


def save_csv_part(upload_dir, part_number, payload, start, stop, values, labels, columns, byte_limit):
    path = upload_dir / f"nidr_test_{part_number:05d}.csv"
    path.write_bytes(payload)
    expected_labels = None if labels is None else labels[start:stop]
    validate_csv(path, values[start:stop], expected_labels, columns, byte_limit)
    return {
        "file": f"uploads/{path.name}", "rows": stop - start,
        "bytes": path.stat().st_size, "sha256": sha256_file(path),
        "test_start_row": start, "test_stop_row_exclusive": stop,
    }


def export_csv_parts(values, labels, upload_dir, start, stop, rows_per_file, byte_limit):
    columns = FEATURE_NAMES + (["label"] if labels is not None else [])
    header = csv_line(columns)
    payload = bytearray(header)
    part_start = start
    records = []
    class_counts = [0] * len(CLASS_NAMES)
    missing_cells = 0
    # Các lát đọc mảng tối đa 1.000 dòng; không nạp toàn bộ test vào DataFrame.
    for chunk_start in range(start, stop, SYSTEM_MAX_ROWS):
        chunk_stop = min(chunk_start + SYSTEM_MAX_ROWS, stop)
        chunk = values[chunk_start:chunk_stop]
        if np.isinf(chunk).any():
            raise ValueError("Test còn Inf. Kiểm tra lại output processing; không tự sửa dữ liệu.")
        missing_cells += int(np.isnan(chunk).sum())
        if labels is not None:
            label_chunk = labels[chunk_start:chunk_stop]
            if np.any(label_chunk < 0) or np.any(label_chunk >= len(CLASS_NAMES)):
                raise ValueError("y_test có class ID ngoài khoảng 0–7.")
        for offset, row in enumerate(chunk):
            row_index = chunk_start + offset
            fields = []
            for value in row:
                fields.append("" if np.isnan(value) else format(float(value), ".17g"))
            if labels is not None:
                class_id = int(labels[row_index])
                fields.append(CLASS_NAMES[class_id])
                class_counts[class_id] += 1
            encoded_row = csv_line(fields)
            if len(header) + len(encoded_row) > byte_limit:
                raise ValueError("Một dòng và header đã vượt giới hạn byte; tăng MAX_FILE_BYTES.")
            row_count = row_index - part_start
            if row_count and (row_count >= rows_per_file or len(payload) + len(encoded_row) > byte_limit):
                record = save_csv_part(
                    upload_dir, len(records) + 1, payload, part_start, row_index,
                    values, labels, columns, byte_limit,
                )
                records.append(record)
                payload = bytearray(header)
                part_start = row_index
            payload.extend(encoded_row)
    if stop > part_start:
        records.append(save_csv_part(
            upload_dir, len(records) + 1, payload, part_start, stop,
            values, labels, columns, byte_limit,
        ))
    if sum(record["rows"] for record in records) != stop - start:
        raise ValueError("Tổng số dòng xuất không khớp vùng test đã chọn.")
    return records, class_counts, missing_cells


def export_test_uploads(data_dir, output_root, include_label=True, rows_per_file=1000,
                        max_file_bytes=SYSTEM_MAX_BYTES, start_row=0, max_total_rows=None,
                        create_zip=True):
    if type(include_label) is not bool or type(create_zip) is not bool:
        raise ValueError("INCLUDE_LABEL và CREATE_ZIP phải là True hoặc False.")
    values, labels, source_info = load_test_source(data_dir, include_label)
    stop = validate_export_options(
        rows_per_file, max_file_bytes, start_row, max_total_rows, len(values),
    )
    run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
    output_dir = Path(output_root) / f"test_upload_{run_id}"
    upload_dir = output_dir / "uploads"
    upload_dir.mkdir(parents=True, exist_ok=False)
    status_path = output_dir / "run_status.json"
    archive_path = output_dir.with_suffix(".zip")
    partial_archive = output_dir.with_suffix(".zip.part")
    status = {"run_id": run_id, "status": "in_progress", "source_run_id": EXPECTED_DATA_RUN_ID}
    write_json(status_path, status)
    try:
        records, class_counts, missing_cells = export_csv_parts(
            values, labels, upload_dir, start_row, stop, rows_per_file, max_file_bytes,
        )
        manifest = pd.DataFrame(records)
        manifest.to_csv(output_dir / "manifest.csv", index=False)
        write_json(output_dir / "export_protocol.json", {
            "source": source_info,
            "test_start_row": start_row, "test_stop_row_exclusive": stop,
            "rows_exported": stop - start_row, "files_exported": len(records),
            "rows_per_file_limit": rows_per_file, "bytes_per_file_limit": max_file_bytes,
            "include_label": include_label, "class_names": CLASS_NAMES,
            "source_label_counts": dict(zip(CLASS_NAMES, class_counts)) if include_label else None,
            "nan_cells_exported": missing_cells,
            "order": "contiguous_test_array_order_no_resampling",
            "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__},
            "checks": {"source_checksums": True, "csv_readback_float32_and_labels": True,
                       "row_and_byte_limits": True, "selected_row_coverage": True},
        })
        write_json(output_dir / "input_schema.json", {
            "feature_names": FEATURE_NAMES, "class_names": CLASS_NAMES,
            "optional_column": "label", "input_scale": "original_float32_nan_retained",
        })
        (output_dir / "README.txt").write_text(
            "CSV đầu vào NIDR — chỉ lấy test của data run " + EXPECTED_DATA_RUN_ID + "\n"
            "Chỉ upload các file trong uploads/. Không upload manifest.csv hoặc ZIP.\n"
            "Giữ nguyên 46 đặc trưng thang đo gốc, NaN là ô trống. BE tự transform bằng bundle đóng băng.\n"
            "label (nếu có) là nhãn nguồn 8 lớp để đối chiếu, không dùng cho model/Gemini.\n"
            "manifest.csv: đoạn [test_start_row, test_stop_row_exclusive) trong mảng test sau chuẩn bị.\n"
            "Baseline validation/test đã từng được xem; không phải đánh giá độc lập mới.\n",
            encoding="utf-8",
        )
        completed_status = dict(
            status, status="complete", files=len(records), rows=stop - start_row,
            finished_at=datetime.now(timezone.utc).isoformat(),
        )
        if create_zip:
            with zipfile.ZipFile(partial_archive, "w", compression=zipfile.ZIP_DEFLATED) as archive:
                for path in sorted(output_dir.rglob("*")):
                    if path.is_file() and path != status_path:
                        archive.write(path, path.relative_to(output_dir).as_posix())
                archive.writestr("run_status.json", json.dumps(completed_status, ensure_ascii=False, indent=2))
            partial_archive.replace(archive_path)
        # Chỉ báo complete sau khi đọc lại CSV và đóng ZIP thành công.
        write_json(status_path, completed_status)
    except Exception as error:
        partial_archive.unlink(missing_ok=True)
        archive_path.unlink(missing_ok=True)
        status.update(status="failed", error=str(error))
        write_json(status_path, status)
        raise
    return output_dir, archive_path if create_zip else None, manifest

## 4. Chạy xuất

Nếu thiếu đúng output processing, notebook dừng tại đây. Không đổi sang CSV raw, train, validation hoặc mảng đã scale để chạy tiếp. Khi lỗi, xem `run_status.json`; không upload file còn lại trong run `failed`.

In [ ]:
if not Path("/kaggle/input").is_dir() or not Path("/kaggle/working").is_dir():
    raise RuntimeError("Hãy chạy notebook này trên Kaggle và gắn đúng output processing.")

selected_data_dir = find_data_dir(DATA_DIR)
print("Nguồn test:", selected_data_dir)
print("Đang kiểm checksum nguồn, chia CSV và kiểm tra đọc lại...")
output_dir, archive_path, manifest = export_test_uploads(
    data_dir=selected_data_dir,
    output_root=OUTPUT_ROOT,
    include_label=INCLUDE_LABEL,
    rows_per_file=ROWS_PER_FILE,
    max_file_bytes=MAX_FILE_BYTES,
    start_row=START_ROW,
    max_total_rows=MAX_TOTAL_ROWS,
    create_zip=CREATE_ZIP,
)
print("Hoàn tất:", output_dir)
print(f"{int(manifest['rows'].sum()):,} dòng test → {len(manifest):,} CSV")
print(f"File lớn nhất: {int(manifest['bytes'].max()):,} byte")
display(manifest.head(10))

In [ ]:
from IPython.display import FileLink, display

protocol = read_json(output_dir / "export_protocol.json")
if INCLUDE_LABEL:
    print("Phân bố nhãn nguồn của phần test đã xuất (không phải dự đoán):")
    display(pd.DataFrame({
        "label": CLASS_NAMES,
        "rows": [protocol["source_label_counts"][name] for name in CLASS_NAMES],
    }))
if archive_path is not None:
    # FileLink dùng đường dẫn tương đối với /kaggle/working để tải trong notebook.
    display(FileLink(str(archive_path.relative_to(Path("/kaggle/working")))))
print("Save Version → Save & Run All; tải ZIP từ Output, giải nén và upload CSV trong uploads/ vào NIDR.")
print("Đối chiếu manifest.csv để biết mỗi CSV ứng với đoạn nào trong test.")